# Digit addition and its carry circuit

Train a small transformer on 2-digit decimal addition, then work out what algorithm it actually
learned.

Nanda et al. did this for modular arithmetic in *Progress measures for grokking via mechanistic
interpretability*, and found the model had built something out of discrete Fourier transforms and
trigonometric identities. Plain decimal addition is a different shape of problem. It's positional
and it needs carrying, so I'd expect a different circuit. Working out how different is the point.

The model is a TransformerLens `HookedTransformer` rather than my own implementation. That costs
nothing here and gives me hook points for the ablations later.

One thing makes this tractable: the vocabulary is 12 symbols, the digits plus `+` and `=`. In a
language model most of the parameters sit in the embedding table and never compute anything. Here
almost every parameter is doing work, so there should be a real algorithm to find rather than a
lookup table.

## The floor

Identify which attention heads implement the carry, using ablation and attention patterns.

Everything past that is bonus: the full mechanism, whether the circuit differs across digit
positions, the grokking dynamics. Interpretability has no natural stopping point, so it's worth
writing down in advance what counts as done.

In [ ]:
from dataclasses import asdict, dataclass

import einops
import torch as t
from jaxtyping import Int
from torch import Tensor
from tqdm.notebook import tqdm
from transformer_lens import HookedTransformer, HookedTransformerConfig

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
print("device:", device)

---

## 1. Task encoding

Sequences look like `49+97=641`, fixed length, so every example has the same shape. Twelve tokens
in the vocabulary: ten digits, `+`, and `=`. A dict covers it. There's no reason to train a
tokenizer for twelve symbols.

Two of the encoding choices matter more than they look.

The answer is written backwards, least significant digit first. Carries then propagate in the same
direction the model generates, so each answer digit depends on things it has already emitted.
Writing `146` the normal way round would ask it to predict the leading digit before it has worked
out the carries that determine it.

Numbers are zero-padded, `05` rather than `5`, so a given position always means the same thing.

Two digits is the smallest size that still needs carrying, and 100 x 100 = 10,000 pairs is small
enough to enumerate the whole problem.

In [ ]:
VOCABULARY = {f"{i:d}": i for i in range(10)}
VOCABULARY["+"] = 10
VOCABULARY["="] = 11

# convenience aliases
TOKEN_TO_ID = VOCABULARY
ID_TO_TOKEN = {idx: token for token, idx in TOKEN_TO_ID.items()}
D_VOCAB = len(VOCABULARY)

# 1D (seq,) or 2D (batch, seq) ids, as a tensor or plain lists
TokenIds = Tensor | list[int] | list[list[int]]


def to_tokens(strings: str | list[str]) -> Int[Tensor, "batch seq"]:
    """'49+97=641' -> tensor([[4, 9, 10, 9, 7, 11, 6, 4, 1]]). Always returns (batch, seq)."""
    if isinstance(strings, str):
        strings = [strings]
    return t.tensor([[TOKEN_TO_ID[c] for c in s] for s in strings], device=device)


def to_str_tokens(tokens: TokenIds) -> list[str] | list[list[str]]:
    """Token ids -> the characters they stand for, one per token. Nested if the input is 2D."""
    if isinstance(tokens, Tensor):
        tokens = tokens.tolist()
    if tokens and isinstance(tokens[0], list):
        return [[ID_TO_TOKEN[i] for i in row] for row in tokens]
    return [ID_TO_TOKEN[i] for i in tokens]


def to_string(tokens: TokenIds) -> str | list[str]:
    """Token ids -> '49+97=641'. A list of strings if the input is 2D."""
    strs = to_str_tokens(tokens)
    if strs and isinstance(strs[0], list):
        return ["".join(row) for row in strs]
    return "".join(strs)


# sequences will have the result of the computation reversed


## 2. Dataset

Enumerate every `(a, b)` pair, then split. 10,000 examples of 9 tokens is one tensor that fits in
memory, so there's no need for `datasets`, Arrow, or DataLoader workers.

The split fraction is the knob that matters most. Holding out a large fraction is what makes
grokking possible: the model memorises the training pairs, sits near chance on everything held
out, then generalises abruptly thousands of steps later. Reproducing that here would be a result
on its own, so the fraction is worth sweeping once a basic run works.

Skipping ahead: it does not reproduce, and the sweep below explains why. Addition decomposes into
only 200 `(digit, digit, carry_in)` table entries, so at Nanda's `frac_train=0.3` generalising is
cheaper than memorising and there is no delayed transition to see. The interesting behaviour is
two orders of magnitude further down, at `frac_train` around 0.01.

Loss is computed only on the answer positions. The question digits are given to the model, so
predicting them is free and would dilute the number.

In [ ]:
def sum_dataset(num_digits: int = 2, seed: int = 42, frac_train: float = 0.3) -> tuple[Tensor, Tensor]:
    
    pairs = []

    for i in range(10**num_digits):
        for j in range(10**num_digits):
            pairs.append(f"{i:0{num_digits}d}+{j:0{num_digits}d}={f'{(i+j):0{num_digits+1}d}'[::-1]}")

    data = t.tensor([[TOKEN_TO_ID[token] for token in example] for example in pairs], device=device)
    g = t.Generator().manual_seed(seed)

    _n = data.shape[0]
    split = t.randperm(_n, generator=g)
    train_idx = split[:int(frac_train * _n)].to(device=device)
    test_idx = split[int(frac_train * _n):].to(device=device)


    assert t.cat([train_idx, test_idx], dim=0).sort().values.equal(t.arange(len(data), device=data.device))

    train = data[train_idx]
    test = data[test_idx]
    
    return train, test
train, test = sum_dataset()
print(f"train: {train.shape} \ntest: {test.shape}")

## 3. Train

Sizing follows Nanda et al. wherever the task allows: one layer, `d_model=128`, 4 heads,
`d_mlp=512`, no layer norm. The deviations are a 12-symbol vocabulary and a 9-token context, both
forced by the task, and AdamW's default `betas=(0.9, 0.999)` where he uses `(0.9, 0.98)`. Keeping
everything else fixed means the task is the only variable between his setup and this one.

Training is full batch, AdamW, constant `lr=1e-3`, `weight_decay=1.0`. That last one isn't
incidental. Grokking depends on the decoupled weight decay specifically, and full batch keeps the
loss curves clean enough to see the transition when it happens.

### Training

I started from ARENA's `TrainArgs` and cut it down. Anything duplicating `cfg` went, since two
places holding the same number will eventually disagree, and so did everything that only makes
sense when batching through a DataLoader. Full batch means one epoch is one optimizer step, so a
single counter covers it.

In [ ]:
@dataclass
class TrainArgs:
    num_digits: int
    n_layers: int
    # trainset_size: int   -- derivable from train.shape[0]
    # valset_size: int     -- derivable from test.shape[0]
    frac_train: float
    epochs: int
    # batch_size: int      -- full batch, so there is nothing to vary
    lr: float
    # lr_start: float
    # lr_end: float
    weight_decay: float
    seed: int
    eval_every: int
    # d_model: int
    # d_head: int
    # n_heads: int
    # d_mlp: int
    # normalization_type: str | None
    use_wandb: bool
    device: str
    wandb_project: str = "transformer-digit-addition"
    wandb_group: str | None = None

    def __post_init__(self):
        self.seq_len = self.num_digits * 3 + 2 + 1  # We have [{a}, +, {b}, =, {a+b}]
        self.run_name = (
            f"L{self.n_layers}_wd{self.weight_decay}"
            f"_ft{self.frac_train}_s{self.seed}"
        )

In [ ]:
args = TrainArgs(
    num_digits=2,
    n_layers=1,
    # 0.01 of 10,000 = 100 examples, exposing 114 of the 200
    # (digit, digit, carry_in) table entries. A model that learns only what it
    # has seen tops out at 0.671 per-digit / 0.443 per-sequence, so there is a
    # wide window between "memorised the table" and "inferred the rest".
    frac_train=0.01,
    # Nanda's protocol, unchanged: constant lr, very high weight decay,
    # 40k steps. Grokking appeared around step 10k in his setup.
    epochs=40000,
    lr=1e-3,
    weight_decay=1.0,
    seed=42,
    # test set is now 9,800 rows, ~49x the training forward - evaluating
    # every 10 steps would dominate the runtime
    eval_every=50,
    # off by default so the notebook runs without a wandb account; set True to log
    use_wandb=False,
    device=device,
    wandb_group="grokking-hunt",
)

### Defining the model

In [ ]:
def create_model(args: TrainArgs) -> HookedTransformer:
    """A fresh model from init, sized from `args`.

    Re-seeded on every call, so a sweep over `frac_train` varies only the thing
    being swept -- every run starts from identical weights.
    """
    t.manual_seed(args.seed)

    d_model = 128
    n_heads = 4

    cfg = HookedTransformerConfig(
        n_layers=args.n_layers,
        n_ctx=args.seq_len,  # We have [{a}, +, {b}, =, {a+b}]
        d_model=d_model,
        d_head=d_model // n_heads,
        n_heads=n_heads,
        d_mlp=4 * d_model,
        attn_only=False,
        act_fn="relu",
        # We have all digits, and "+="
        d_vocab=len(VOCABULARY),
        # off for training: these materialise per-head tensors and cost ~140x
        # throughput. Switch on before the interpretability section.
        use_attn_result=False,
        use_split_qkv_input=False,
        use_hook_tokens=False,
        normalization_type=None,
        device=args.device,
    )

    return HookedTransformer(cfg)

model = create_model(args)
cfg = model.cfg

In [ ]:
def param_breakdown(model: HookedTransformer, n_train: int | None = None) -> None:
    """Parameter counts grouped by component, with the attention:MLP ratio.

    The ratio is worth watching more than the total -- attention routes information
    between positions, MLPs do the arithmetic, and a config that starves the MLPs
    will struggle on this task. The standard d_mlp = 4 * d_model gives ~0.50x.
    """
    groups: dict[str, int] = {}
    for name, param in model.named_parameters():
        key = "attn" if ".attn." in name else "mlp" if ".mlp." in name else "embed/unembed"
        groups[key] = groups.get(key, 0) + param.numel()

    total = sum(groups.values())
    for key, count in sorted(groups.items(), key=lambda kv: -kv[1]):
        print(f"  {key:16s} {count:>10,}  ({count / total * 100:5.1f}%)")
    print(f"  {'total':16s} {total:>10,}")

    if groups.get("mlp"):
        print(f"\n  attn:mlp ratio   {groups['attn'] / groups['mlp']:.2f}x")
    if n_train:
        print(f"  params per training example   {total / n_train:.0f}")


param_breakdown(model)


In [ ]:
for name, param in model.named_parameters():
    print(f"{name:<20} {str(param.shape):<20} {param.numel()}")
sum(p.numel() for p in model.parameters())      # 584,556

In [ ]:
print(
    train.shape,
    to_string(train[t.randint(low=0, high=len(train), size=(1,))]),
    sep="\n",
)

In [ ]:
class Trainer:
    def __init__(self, model: HookedTransformer, args: TrainArgs):
        self.args = args
        self.model = model
        
    def training_step(self, toks: Tensor) -> Tensor:
        logits, target = self._shared_train_validation_step(toks)
        return self._loss(logits, target)
    
    def validation_step(self, toks: Tensor) -> tuple[float, float, float]:
        """Loss, per-digit accuracy, per-sequence accuracy.

        Per-sequence is the metric that matters: a sum with one wrong digit is a
        wrong sum. It also separates "learned the seen table entries" from "full
        generalisation" far more sharply than the per-digit average does.
        """
        logits, target = self._shared_train_validation_step(toks)
        loss = self._loss(logits, target).item()
        correct = logits.argmax(dim=-1) == target
        return loss, correct.float().mean().item(), correct.all(dim=-1).float().mean().item()
    
    def _loss(self, logits: Tensor, target: Tensor) -> Tensor:
        return t.nn.functional.cross_entropy(
            einops.rearrange(logits, "batch seq vocab_out -> (batch seq) vocab_out"),
            einops.rearrange(target, "batch seq -> (batch seq)")
        )
    
    def _shared_train_validation_step(self, toks: Tensor) -> tuple[Tensor, Tensor]:
        toks = toks.to(self.args.device)
        logits = self.model(toks)[:, -(self.args.num_digits + 2): -1]
        target = toks[:, -(self.args.num_digits + 1):]
        return logits, target
    
    def configure_optimizers(self):
        optimizer = t.optim.AdamW(
            self.model.parameters(), lr=self.args.lr, weight_decay=self.args.weight_decay
        )
        return optimizer

def train_model(model: HookedTransformer, args: TrainArgs):
    if args.use_wandb:
        import wandb
        configs = asdict(args)
        configs["device"] = str(args.device)
        wandb.init(
            project=args.wandb_project,
            group=args.wandb_group,
            name=args.run_name,
            config=configs,
        )
    trainer = Trainer(model=model, args=args)
    optimizer = trainer.configure_optimizers()
    
    train, test = sum_dataset(
        num_digits=args.num_digits, seed=args.seed, frac_train=args.frac_train
    )

    # the two curves that make the grokking plot, plus test accuracy
    history = {
        "step": [],
        "train_loss": [],
        "test_loss": [],
        "test_accuracy": [],
        "test_seq_accuracy": [],
        # grokking is defined by the gap between train and test, so log both
        "train_accuracy": [],
        "train_seq_accuracy": [],
    }

    # training
    optimizer.zero_grad()
    progress_bar = tqdm(range(args.epochs))
    try:
      for epoch in progress_bar:
        loss = trainer.training_step(train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
        # validation
        if epoch % args.eval_every == 0 or epoch == args.epochs - 1:
            with t.inference_mode():
                test_loss, test_accuracy, test_seq = trainer.validation_step(test)
                _, train_accuracy, train_seq = trainer.validation_step(train)
            if args.use_wandb:
                wandb.log(
                    {
                        "train_loss": loss.item(),
                        "test_loss": test_loss,
                        "test_accuracy": test_accuracy,
                        "test_seq_accuracy": test_seq,
                        "train_accuracy": train_accuracy,
                        "train_seq_accuracy": train_seq,
                    },
                    step=epoch,
                )
            history["step"].append(epoch)
            history["train_loss"].append(loss.item())
            history["test_loss"].append(test_loss)
            history["test_accuracy"].append(test_accuracy)
            history["test_seq_accuracy"].append(test_seq)
            history["train_accuracy"].append(train_accuracy)
            history["train_seq_accuracy"].append(train_seq)
            progress_bar.set_description(
                f"train {loss.item():.4f} | test {test_loss:.4f} | "
                f"seq {test_seq:.3f} | digit {test_accuracy:.3f}"
            )
    except KeyboardInterrupt:
        print(f"interrupted at step {epoch}")
    finally:
        if args.use_wandb:
            wandb.finish()

    return trainer.model, history


In [ ]:
model = create_model(args)
model, history = train_model(model, args)

In [ ]:
import json, numpy as np

# NB: overwrites the committed history for this run_name if you rerun with fewer steps
json.dump(history, open(f"../results/history_{args.run_name}.json", "w"))

s   = np.array(history["step"])
tl  = np.array(history["train_loss"])
vl  = np.array(history["test_loss"])
acc = np.array(history["test_accuracy"])

solved = np.where(acc > 0.99)[0]
# a spike has to be relative: these runs sit at ~1e-6 train loss, so any fixed threshold
# catches everything or nothing. Same 10x rule as carry_circuit.ipynb.
spikes = int((tl[1:] / np.maximum(tl[:-1], 1e-12) > 10).sum())

print(f"saved  ../results/history_{args.run_name}.json")
print(f"final: train {tl[-1]:.4f} | test {vl[-1]:.4f} | acc {acc[-1]:.4f}")
print(f"max train loss {tl.max():.3f} at step {s[tl.argmax()]}")
print(f"spikes (>10x jump in train loss): {spikes}")
print(f"first solved at step {s[solved[0]] if len(solved) else 'never'}")
print(f"fraction of evals solved: {100*(acc > 0.99).mean():.1f}%")
print(f"max |train - test| loss gap: {np.abs(tl - vl).max():.4f}")


In [ ]:
import glob
import json
import os
import re

import matplotlib.pyplot as plt
import numpy as np

RESULTS = "../results"

SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_2 = "#52514e"
GRID = "#e3e2dd"

# colour follows the entity: a fixed wd -> slot map, so adding a run never repaints
# the ones already plotted
WD_COLOUR = {0.0: "#2a78d6", 0.01: "#eb6834", 0.1: "#1baf7a", 1.0: "#eda100"}

# accuracy levels predicted analytically for "solves k of 3 answer positions,
# marginal on the rest"
# per-digit reference levels: what accuracy a model scores if it solves k of the
# 3 answer positions and predicts the marginal on the rest
ACC_LEVELS = [(0.235, "none"), (0.400, "c3 only"), (0.700, "two of three"), (1.0, "all three")]


def load(layers=None):
    """Every saved run, as (weight_decay, label, history). Filter by layer count so
    a 1-layer and a 2-layer run at the same wd never share a line."""
    runs = []
    for path in sorted(glob.glob(f"{RESULTS}/history_*.json")):
        name = os.path.basename(path)[len("history_") : -len(".json")]
        wd = float(re.search(r"wd([0-9.]+)", name).group(1))
        L = int(m.group(1)) if (m := re.search(r"L(\d+)", name)) else 2
        if layers is not None and L != layers:
            continue
        runs.append((wd, name, json.load(open(path))))
    return sorted(runs, key=lambda r: r[0])


def plot(runs, outfile, title=None):
    if not runs:
        raise ValueError(
            f"no runs to plot in {RESULTS} - check the layer filter, and that the "
            f"run actually reached the save cell"
        )
    fig, (ax_loss, ax_acc) = plt.subplots(
        2, 1, figsize=(9, 7), sharex=True, gridspec_kw={"height_ratios": [1.3, 1]}
    )
    fig.patch.set_facecolor(SURFACE)

    for ax in (ax_loss, ax_acc):
        ax.set_facecolor(SURFACE)
        ax.grid(True, color=GRID, linewidth=0.6, zorder=0)
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_color(GRID)
        ax.tick_params(colors=INK_2, labelsize=9)

    for wd, label, h in runs:
        c = WD_COLOUR.get(wd, "#8b8a85")
        s = np.array(h["step"])
        ax_loss.plot(s, h["test_loss"], color=c, linewidth=2, zorder=3, label=label)
        ax_loss.plot(s, h["train_loss"], color=c, linewidth=1, linestyle=(0, (3, 2)), alpha=0.75, zorder=2)
        # solid = per-sequence (every digit right), dashed = per-digit average.
        # older runs predate per-sequence logging, so fall back gracefully.
        seq = h.get("test_seq_accuracy")
        if seq:
            ax_acc.plot(s, seq, color=c, linewidth=2, zorder=3)
            ax_acc.plot(s, h["test_accuracy"], color=c, linewidth=1,
                        linestyle=(0, (3, 2)), alpha=0.75, zorder=2)
        else:
            ax_acc.plot(s, h["test_accuracy"], color=c, linewidth=1,
                        linestyle=(0, (3, 2)), alpha=0.75, zorder=2)
        # direct label at the end of each line (also the relief for the contrast WARN)
        ax_acc.annotate(
            f"wd {wd:g}",
            xy=(s[-1], (h.get("test_seq_accuracy") or h["test_accuracy"])[-1]),
            xytext=(-4, 10),
            textcoords="offset points",
            ha="right",
            va="bottom",
            fontsize=9,
            color=INK_2,
        )

    ax_loss.set_yscale("log")
    ax_loss.set_ylabel("cross-entropy loss", color=INK_2, fontsize=10)
    ax_loss.set_title(
        title or "loss and accuracy by weight decay",
        color=INK,
        fontsize=13,
        loc="left",
        pad=12,
    )
    ax_loss.axhline(1.766, color=INK_2, linewidth=0.9, linestyle=":", zorder=1)
    ax_loss.annotate(
        "marginal-only (1.77)",
        xy=(0.995, 1.766),
        xycoords=("axes fraction", "data"),
        ha="right",
        va="bottom",
        fontsize=8,
        color=INK_2,
    )
    leg = ax_loss.legend(frameon=False, fontsize=9, labelcolor=INK_2, loc="lower left")
    leg.set_title("solid = test, dashed = train", prop={"size": 8})
    leg.get_title().set_color(INK_2)

    for y, lbl in ACC_LEVELS:
        ax_acc.axhline(y, color=GRID, linewidth=0.9, linestyle=":", zorder=1)
        ax_acc.annotate(
            lbl, xy=(0.002, y), xycoords=("axes fraction", "data"),
            va="bottom", fontsize=8, color=INK_2,
        )
    ax_acc.set_ylim(0, 1.12)
    ax_acc.set_ylabel(
        "test accuracy\n(solid = per-sequence, dashed = per-digit)",
        color=INK_2, fontsize=9,
    )
    ax_acc.set_xlabel("optimizer step", color=INK_2, fontsize=10)

    fig.tight_layout()
    fig.savefig(outfile, dpi=200, facecolor=SURFACE)
    print("wrote", outfile)


runs = load(layers=args.n_layers)
for wd, label, h in runs:
    print(f"{label}: {len(h['step'])} points, to step {h['step'][-1]}")
plot(
    runs,
    f"{RESULTS}/runs_L{args.n_layers}.png",
    title=f"{args.n_layers}-layer, {args.num_digits}-digit: training runs",
)


## 4. Interpretability

Lives in `carry_circuit.ipynb`, which loads a saved
checkpoint instead of retraining. Plan:

Roughly in order of cost:

- Attention patterns on carry versus non-carry examples, via `circuitsvis`. Do particular heads
  look at the digit positions that generate a carry?
- Per-head ablation. Zero `hook_z` for one head at a time and measure the accuracy drop. That
  tells me which heads matter, not how they work.
- Direct logit attribution, to find which component actually writes the answer digit.
- Then the mechanism: how carry information moves between digit positions.

Method reference is ARENA's exercise on the Nanda result, *1.5.2 Grokking & Modular Arithmetic*.

The obvious trap is attention sinks. These sequences have no BOS token, so a head with nothing
useful to attend to will dump its attention on whatever sits at position 0, or on the `=`. If a
head looks like it attends to the equals sign for no reason I can explain, it's probably a sink
and not part of any circuit.

## 5. Write-up

Goes in `README.md` and `results/`: motivation, method, findings, plots. The comparison that
matters is the circuit here against Nanda's modular-arithmetic one.